In [1]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = False    # <--- Set True to remove existing environment

opts_str = " ".join(
  [o for c, o in [(USE_COMPLEX, "--complex"), (USE_CLEAN, "--clean")] if c]
)

get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}")

🔧 FEniCSx Setup Configuration
PETSc type      : real
Clean install   : False

⚠️  Google Drive not mounted — using local cache (/content)

🔧 Installing FEniCSx environment...

🔍 Verifying PETSc type...
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered

✅ FEniCSx setup complete!

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: Real PETSc is installed
   - Recommended for most FEM problems
   - For complex problems, reinstall with --complex


---

In [2]:
%%fenicsx

"""
FFCx Code Generation Examples
==============================

This script demonstrates the key concepts from the FEniCS Workshop slides
on generating code for assembling tensors using FFCx.

Examples included:
1. Simple mass matrix (bilinear form)
2. Laplacian problem
3. Poisson equation with source term
4. Vector Laplacian
5. Mixed finite element form

Author: FEniCS Workshop
Date: 2024
"""

import basix.ufl
import ufl

# ============================================================================
# Example 1: Simple Mass Matrix (Inner Product)
# ============================================================================
print("=" * 70)
print("Example 1: Simple Mass Matrix")
print("=" * 70)

# Define the cell type
cell = "triangle"

# Create coordinate element for the mesh (2D)
c_el = basix.ufl.element("Lagrange", cell, 1, shape=(2,))

# Define the domain (mesh)
domain = ufl.Mesh(c_el)

# Define function space with P2 Lagrange elements
el = basix.ufl.element("Lagrange", cell, 2)
V = ufl.FunctionSpace(domain, el)

# Define trial and test functions
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Define the bilinear form: mass matrix
# This corresponds to: ∫ u * v dx
a_mass = ufl.inner(u, v) * ufl.dx

print("Bilinear form (mass matrix):")
print(f"  a = ∫ u * v dx")
print(f"  Function space: P2 Lagrange on triangles")
print(f"  Number of local DOFs: 6 (for P2 triangle)")

# ============================================================================
# Example 2: Laplacian Problem (Stiffness Matrix)
# ============================================================================
print("\n" + "=" * 70)
print("Example 2: Laplacian Problem")
print("=" * 70)

# Use P1 elements for this example
el_p1 = basix.ufl.element("Lagrange", cell, 1)
V_p1 = ufl.FunctionSpace(domain, el_p1)

u_p1 = ufl.TrialFunction(V_p1)
v_p1 = ufl.TestFunction(V_p1)

# Define the bilinear form: stiffness matrix
# This corresponds to: ∫ ∇u · ∇v dx
a_laplacian = ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * ufl.dx

print("Bilinear form (Laplacian):")
print(f"  a = ∫ ∇u · ∇v dx")
print(f"  Function space: P1 Lagrange on triangles")
print(f"  Number of local DOFs: 3 (for P1 triangle)")

# ============================================================================
# Example 3: Poisson Equation with Source Term
# ============================================================================
print("\n" + "=" * 70)
print("Example 3: Poisson Equation")
print("=" * 70)

# Define coefficient function (source term)
f = ufl.Coefficient(V_p1)

# Bilinear form (same as Laplacian)
a_poisson = ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * ufl.dx

# Linear form (right-hand side)
# This corresponds to: ∫ f * v dx
L_poisson = f * v_p1 * ufl.dx

print("Poisson equation: -∇²u = f")
print(f"  Bilinear form: a = ∫ ∇u · ∇v dx")
print(f"  Linear form: L = ∫ f * v dx")
print(f"  Function space: P1 Lagrange on triangles")

# ============================================================================
# Example 4: Vector Laplacian (Elasticity-like)
# ============================================================================
print("\n" + "=" * 70)
print("Example 4: Vector Laplacian")
print("=" * 70)

# Define vector function space (2D vector field)
el_vec = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
V_vec = ufl.FunctionSpace(domain, el_vec)

u_vec = ufl.TrialFunction(V_vec)
v_vec = ufl.TestFunction(V_vec)

# Vector Laplacian: ∫ ∇u : ∇v dx
a_vec_laplacian = ufl.inner(ufl.grad(u_vec), ufl.grad(v_vec)) * ufl.dx

print("Vector Laplacian:")
print(f"  a = ∫ ∇u : ∇v dx")
print(f"  Function space: P1 Lagrange vector field on triangles")
print(f"  Vector dimension: 2")

# ============================================================================
# Example 5: Mixed Formulation (Saddle Point Problem)
# ============================================================================
print("\n" + "=" * 70)
print("Example 5: Mixed Formulation")
print("=" * 70)

# Define two function spaces
V_vel = ufl.FunctionSpace(domain, el_vec)  # Velocity (vector)
V_p = ufl.FunctionSpace(domain, el_p1)     # Pressure (scalar)

# Define trial and test functions
u_vel = ufl.TrialFunction(V_vel)
v_vel = ufl.TestFunction(V_vel)
p = ufl.TrialFunction(V_p)
q = ufl.TestFunction(V_p)

# Mixed bilinear form (simplified Stokes-like)
# a((u,p), (v,q)) = ∫ ∇u:∇v dx - ∫ p div(v) dx - ∫ q div(u) dx
a_mixed_11 = ufl.inner(ufl.grad(u_vel), ufl.grad(v_vel)) * ufl.dx
a_mixed_12 = -p * ufl.div(v_vel) * ufl.dx
a_mixed_21 = -q * ufl.div(u_vel) * ufl.dx

print("Mixed formulation (Stokes-like):")
print(f"  a₁₁ = ∫ ∇u : ∇v dx")
print(f"  a₁₂ = -∫ p div(v) dx")
print(f"  a₂₁ = -∫ q div(u) dx")

# ============================================================================
# Example 6: Different Element Degrees
# ============================================================================
print("\n" + "=" * 70)
print("Example 6: Comparing Different Element Degrees")
print("=" * 70)

element_degrees = [1, 2, 3, 4]
local_dofs = {
    1: 3,   # P1 triangle: 3 vertices
    2: 6,   # P2 triangle: 3 vertices + 3 edges
    3: 10,  # P3 triangle: 3 vertices + 6 edges + 1 interior
    4: 15,  # P4 triangle: 3 vertices + 9 edges + 3 interior
}

for degree in element_degrees:
    el_deg = basix.ufl.element("Lagrange", cell, degree)
    V_deg = ufl.FunctionSpace(domain, el_deg)
    u_deg = ufl.TrialFunction(V_deg)
    v_deg = ufl.TestFunction(V_deg)

    a_deg = ufl.inner(ufl.grad(u_deg), ufl.grad(v_deg)) * ufl.dx

    print(f"\nP{degree} Lagrange elements:")
    print(f"  Local DOFs per triangle: {local_dofs[degree]}")
    print(f"  Local matrix size: {local_dofs[degree]} × {local_dofs[degree]}")

# ============================================================================
# Example 7: Non-linear Form (for Newton solver)
# ============================================================================
print("\n" + "=" * 70)
print("Example 7: Non-linear Form")
print("=" * 70)

# Current solution estimate
u_n = ufl.Coefficient(V_p1)

# Non-linear residual: F(u) = ∫ ∇u · ∇v dx + ∫ u³ * v dx - ∫ f * v dx
F_nonlinear = (ufl.inner(ufl.grad(u_n), ufl.grad(v_p1)) * ufl.dx +
               u_n**3 * v_p1 * ufl.dx -
               f * v_p1 * ufl.dx)

# Jacobian (derivative of F with respect to u_n)
# This is computed automatically by UFL
J_nonlinear = ufl.derivative(F_nonlinear, u_n, u_p1)

print("Non-linear problem: -∇²u + u³ = f")
print(f"  Residual form: F(u; v) = ∫ ∇u·∇v dx + ∫ u³*v dx - ∫ f*v dx")
print(f"  Jacobian form: J(u; δu, v) = derivative of F w.r.t. u")

# ============================================================================
# Example 8: Time-dependent Problem
# ============================================================================
print("\n" + "=" * 70)
print("Example 8: Time-dependent Problem (Heat Equation)")
print("=" * 70)

# Previous time step solution
u_old = ufl.Coefficient(V_p1)

# Time step
dt = ufl.Constant(domain)

# Backward Euler scheme: (u - u_old)/dt + ∇²u = f
# Weak form: ∫ u*v dx + dt*∫ ∇u·∇v dx = ∫ u_old*v dx + dt*∫ f*v dx

a_heat = (u_p1 * v_p1 * ufl.dx +
          dt * ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * ufl.dx)

L_heat = u_old * v_p1 * ufl.dx + dt * f * v_p1 * ufl.dx

print("Heat equation: ∂u/∂t - ∇²u = f")
print(f"  Bilinear form: a = ∫ u*v dx + Δt*∫ ∇u·∇v dx")
print(f"  Linear form: L = ∫ u_old*v dx + Δt*∫ f*v dx")
print(f"  Time discretization: Backward Euler")

# ============================================================================
# Example 9: Subdomain Integration
# ============================================================================
print("\n" + "=" * 70)
print("Example 9: Integration over Subdomains")
print("=" * 70)

# Define measures for different subdomains
dx_1 = ufl.dx(1)  # Subdomain 1
dx_2 = ufl.dx(2)  # Subdomain 2

# Different coefficients on different subdomains
k1 = ufl.Constant(domain)  # Diffusion coefficient in subdomain 1
k2 = ufl.Constant(domain)  # Diffusion coefficient in subdomain 2

a_subdomain = (k1 * ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * dx_1 +
               k2 * ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * dx_2)

print("Multi-material diffusion problem:")
print(f"  a = ∫_Ω₁ k₁*∇u·∇v dx + ∫_Ω₂ k₂*∇u·∇v dx")
print(f"  Different diffusion coefficients on different subdomains")

# ============================================================================
# Example 10: Boundary Integration
# ============================================================================
print("\n" + "=" * 70)
print("Example 10: Boundary Integration (Neumann BC)")
print("=" * 70)

# Boundary measure
ds = ufl.ds

# Neumann boundary condition coefficient
g = ufl.Coefficient(V_p1)

# Bilinear form (same as before)
a_neumann = ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * ufl.dx

# Linear form with Neumann boundary term
L_neumann = f * v_p1 * ufl.dx + g * v_p1 * ds

print("Poisson with Neumann BC: -∇²u = f in Ω, ∂u/∂n = g on ∂Ω")
print(f"  Bilinear form: a = ∫_Ω ∇u·∇v dx")
print(f"  Linear form: L = ∫_Ω f*v dx + ∫_∂Ω g*v ds")

# ============================================================================
# Collect all forms for FFCx compilation
# ============================================================================
print("\n" + "=" * 70)
print("Summary: Forms for FFCx Compilation")
print("=" * 70)

# Create a list of all forms that should be compiled by FFCx
forms = [
    a_mass,           # Example 1: Mass matrix
    a_laplacian,      # Example 2: Laplacian
    a_poisson,        # Example 3: Poisson bilinear form
    L_poisson,        # Example 3: Poisson linear form
    a_vec_laplacian,  # Example 4: Vector Laplacian
    a_mixed_11,       # Example 5: Mixed form blocks
    a_mixed_12,
    a_mixed_21,
    F_nonlinear,      # Example 7: Non-linear residual
    J_nonlinear,      # Example 7: Jacobian
    a_heat,           # Example 8: Heat equation bilinear
    L_heat,           # Example 8: Heat equation linear
    a_subdomain,      # Example 9: Subdomain integration
    a_neumann,        # Example 10: Neumann BC bilinear
    L_neumann,        # Example 10: Neumann BC linear
]

print(f"\nTotal number of forms to compile: {len(forms)}")
print("\nTo compile these forms with FFCx, run:")
print("  python3 -m ffcx.main --visualise ffcx_examples.py")
print("\nThis will generate:")
print("  - ffcx_examples.c  (C implementation)")
print("  - ffcx_examples.h  (Header file)")
print("  - Computational graph visualizations (if --visualise is used)")

print("\n" + "=" * 70)
print("Examples Complete!")
print("=" * 70)

Example 1: Simple Mass Matrix
Bilinear form (mass matrix):
  a = ∫ u * v dx
  Function space: P2 Lagrange on triangles
  Number of local DOFs: 6 (for P2 triangle)

Example 2: Laplacian Problem
Bilinear form (Laplacian):
  a = ∫ ∇u · ∇v dx
  Function space: P1 Lagrange on triangles
  Number of local DOFs: 3 (for P1 triangle)

Example 3: Poisson Equation
Poisson equation: -∇²u = f
  Bilinear form: a = ∫ ∇u · ∇v dx
  Linear form: L = ∫ f * v dx
  Function space: P1 Lagrange on triangles

Example 4: Vector Laplacian
Vector Laplacian:
  a = ∫ ∇u : ∇v dx
  Function space: P1 Lagrange vector field on triangles
  Vector dimension: 2

Example 5: Mixed Formulation
Mixed formulation (Stokes-like):
  a₁₁ = ∫ ∇u : ∇v dx
  a₁₂ = -∫ p div(v) dx
  a₂₁ = -∫ q div(u) dx

Example 6: Comparing Different Element Degrees

P1 Lagrange elements:
  Local DOFs per triangle: 3
  Local matrix size: 3 × 3

P2 Lagrange elements:
  Local DOFs per triangle: 6
  Local matrix size: 6 × 6

P3 Lagrange elements:
  Local 

---

In [3]:
%%fenicsx

#!/usr/bin/env python3
"""
FFCx Compilation Demo (Colab Compatible)
=========================================

This script demonstrates how to compile UFL forms using FFCx and
inspect the generated code. Works in Google Colab and local environments.

Usage:
    python3 demo_ffcx_compilation_colab.py
"""

import os
import sys
from pathlib import Path
import subprocess

def print_header(title):
    """Print a formatted header."""
    width = 70
    print("\n" + "=" * width)
    print(title.center(width))
    print("=" * width + "\n")

def run_command(cmd, description):
    """Run a shell command and print the output."""
    print(f"\n>>> {description}")
    print(f"Command: {cmd}\n")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode == 0:
        print(result.stdout)
        if result.stderr:
            print("Warnings/Info:")
            print(result.stderr)
    else:
        print(f"Error (exit code {result.returncode}):")
        print(result.stderr)

    return result.returncode == 0

def find_example_file():
    """Find the example file in various possible locations."""

    # Try common locations
    possible_paths = [
        Path.cwd() / "ffcx_examples.py",           # Current directory
        Path(__file__).parent / "ffcx_examples.py", # Same dir as script
        Path("/content/ffcx_examples.py"),          # Colab default
        Path.home() / "ffcx_examples.py",           # Home directory
    ]

    for path in possible_paths:
        if path.exists():
            return path

    return None

def create_example_file_if_missing():
    """Create the example file if it doesn't exist."""

    example_file = Path.cwd() / "ffcx_examples.py"

    if example_file.exists():
        return example_file

    print("Creating ffcx_examples.py in current directory...")

    example_code = '''"""
FFCx Code Generation Examples
==============================
"""

import basix.ufl
import ufl

# Example 1: Simple Mass Matrix
cell = "triangle"
c_el = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
domain = ufl.Mesh(c_el)

el = basix.ufl.element("Lagrange", cell, 2)
V = ufl.FunctionSpace(domain, el)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Bilinear form: mass matrix
a_mass = ufl.inner(u, v) * ufl.dx

# Example 2: Laplacian
el_p1 = basix.ufl.element("Lagrange", cell, 1)
V_p1 = ufl.FunctionSpace(domain, el_p1)

u_p1 = ufl.TrialFunction(V_p1)
v_p1 = ufl.TestFunction(V_p1)

a_laplacian = ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * ufl.dx

# Example 3: Poisson equation
f = ufl.Coefficient(V_p1)

a_poisson = ufl.inner(ufl.grad(u_p1), ufl.grad(v_p1)) * ufl.dx
L_poisson = f * v_p1 * ufl.dx

# Collect all forms
forms = [a_mass, a_laplacian, a_poisson, L_poisson]

print("FFCx examples loaded successfully!")
print(f"Number of forms: {len(forms)}")
'''

    with open(example_file, 'w') as f:
        f.write(example_code)

    print(f"✓ Created {example_file}")
    return example_file

def main():
    """Main demonstration function."""

    print_header("FFCx Code Generation Demonstration (Colab)")

    # Check if FFCx is installed
    try:
        import ffcx
        import ffcx.main
        print("✓ FFCx is installed")
        version = getattr(ffcx, '__version__', 'unknown')
        print(f"  FFCx version: {version}")
    except ImportError:
        print("✗ FFCx is not installed!")
        print("\n  Installing FFCx...")
        result = subprocess.run(
            ["pip", "install", "-q", "fenics-ffcx", "fenics-basix", "fenics-ufl"],
            capture_output=True
        )
        if result.returncode == 0:
            print("  ✓ FFCx installed successfully!")
            import ffcx
            import ffcx.main
        else:
            print("  ✗ Installation failed!")
            return 1

    # Check for basix and ufl
    try:
        import basix.ufl
        import ufl
        print("✓ Basix and UFL are available")
    except ImportError as e:
        print(f"✗ Missing dependencies: {e}")
        print("\n  Installing dependencies...")
        subprocess.run(["pip", "install", "-q", "fenics-basix", "fenics-ufl"])
        import basix.ufl
        import ufl
        print("  ✓ Dependencies installed!")

    # Find or create the example file
    example_file = find_example_file()

    if example_file is None:
        print("\n✗ Example file not found in common locations")
        print("  Creating example file...")
        example_file = create_example_file_if_missing()
    else:
        print(f"✓ Example file found: {example_file}")

    # Change to the directory containing the example file
    os.chdir(example_file.parent)
    print(f"✓ Working directory: {Path.cwd()}")

    # Step 1: Show the example file content
    print_header("Step 1: Preview Example Code")

    with open(example_file, 'r') as f:
        lines = f.readlines()
        preview_lines = min(30, len(lines))
        print(f"Showing first {preview_lines} lines:\n")
        print(''.join(lines[:preview_lines]))
        if len(lines) > preview_lines:
            print(f"\n... ({len(lines) - preview_lines} more lines)")

    # Step 2: Run the example to see the forms
    print_header("Step 2: Execute Example to Define Forms")

    success = run_command(
        f"python3 {example_file.name}",
        "Running the example script"
    )

    if not success:
        print("\n✗ Failed to run example script")
        return 1

    # Step 3: Compile with FFCx
    print_header("Step 3: Compile Forms with FFCx")

    output_dir = example_file.parent

    print("Compiling forms (this may take a moment)...")

    try:
        # Import after potential installation
        import ffcx.main

        ffcx.main.main([
            "-o", str(output_dir),
            str(example_file)
        ])
        print("\n✓ FFCx compilation completed successfully!")
    except Exception as e:
        print(f"\n✗ FFCx compilation failed: {e}")
        import traceback
        traceback.print_exc()
        return 1

    # Step 4: List generated files
    print_header("Step 4: Generated Files")

    c_file = output_dir / f"{example_file.stem}.c"
    h_file = output_dir / f"{example_file.stem}.h"

    generated_files = []
    if c_file.exists():
        generated_files.append(c_file)
    if h_file.exists():
        generated_files.append(h_file)

    if generated_files:
        print("Generated files:")
        for f in sorted(generated_files):
            size = f.stat().st_size
            print(f"  {f.name:<30} ({size:,} bytes)")
    else:
        print("No files were generated!")
        return 1

    # Step 5: Inspect the generated C header
    print_header("Step 5: Inspect Generated Header File")

    if h_file.exists():
        with open(h_file, 'r') as f:
            lines = f.readlines()[:30]
            print("First 30 lines of header file:\n")
            print(''.join(lines))
    else:
        print(f"Header file not found: {h_file}")

    # Step 6: Inspect a generated function signature
    print_header("Step 6: Inspect Assembly Function")

    if c_file.exists():
        with open(c_file, 'r') as f:
            content = f.read()

        # Find first tabulate_tensor function
        import re
        pattern = r'(void tabulate_tensor[^{]+\{[^}]{0,500})'
        match = re.search(pattern, content, re.DOTALL)

        if match:
            print("First assembly function signature:\n")
            snippet = match.group(1)[:500]
            print(snippet)
            if len(match.group(1)) > 500:
                print("\n... (truncated)")
        else:
            print("No tabulate_tensor function found")
    else:
        print(f"C file not found: {c_file}")

    # Step 7: Show quadrature information
    print_header("Step 7: Quadrature Rules in Generated Code")

    if c_file.exists():
        with open(c_file, 'r') as f:
            content = f.read()

        import re
        pattern = r'static const double weights_\w+\[\d+\]\s*=\s*\{[^}]+\}'
        matches = re.findall(pattern, content)

        if matches:
            print(f"Found {len(matches)} quadrature rule(s)\n")
            print("First quadrature rule:")
            print(matches[0][:200])
            if len(matches[0]) > 200:
                print("... (truncated)")
        else:
            print("No quadrature rules found")

    # Step 8: Show statistics
    print_header("Step 8: Code Generation Statistics")

    if c_file.exists() and h_file.exists():
        with open(c_file, 'r') as f:
            c_lines = len(f.readlines())

        with open(h_file, 'r') as f:
            h_lines = len(f.readlines())

        print(f"Generated C code statistics:")
        print(f"  C source file:   {c_lines:,} lines")
        print(f"  Header file:     {h_lines:,} lines")
        print(f"  Total:           {c_lines + h_lines:,} lines")

        # Count functions
        with open(c_file, 'r') as f:
            content = f.read()

        num_functions = content.count('void tabulate_tensor')
        print(f"  Assembly functions: {num_functions}")

    # Final summary
    print_header("Demonstration Complete!")

    print("Summary of what we did:")
    print("  1. ✓ Verified FFCx installation")
    print("  2. ✓ Defined variational forms in Python (UFL)")
    print("  3. ✓ Compiled forms to C code using FFCx")
    print("  4. ✓ Inspected generated assembly code")
    print("  5. ✓ Examined quadrature rules and optimizations")

    print(f"\nGenerated files are in: {output_dir}")
    print(f"  - {c_file.name}")
    print(f"  - {h_file.name}")

    print("\nNext steps:")
    print("  - Study the generated C code in detail")
    print("  - Modify the example forms and recompile")
    print("  - Use the compiled forms in a DOLFINx simulation")

    return 0

if __name__ == "__main__":
    sys.exit(main())


              FFCx Code Generation Demonstration (Colab)              

✓ FFCx is installed
  FFCx version: 0.10.0
✓ Basix and UFL are available

✗ Example file not found in common locations
  Creating example file...
Creating ffcx_examples.py in current directory...
✓ Created /content/ffcx_examples.py
✓ Working directory: /content

                     Step 1: Preview Example Code                     

Showing first 30 lines:

"""
FFCx Code Generation Examples
"""

import basix.ufl
import ufl

# Example 1: Simple Mass Matrix
cell = "triangle"
c_el = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
domain = ufl.Mesh(c_el)

el = basix.ufl.element("Lagrange", cell, 2)
V = ufl.FunctionSpace(domain, el)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Bilinear form: mass matrix
a_mass = ufl.inner(u, v) * ufl.dx

# Example 2: Laplacian
el_p1 = basix.ufl.element("Lagrange", cell, 1)
V_p1 = ufl.FunctionSpace(domain, el_p1)

u_p1 = ufl.TrialFunction(V_p1)
v_p1 = ufl.TestFunction(V_p1)



---

In [4]:
%%fenicsx

#!/usr/bin/env python3
"""
FFCx Code Analysis Tool (Colab Compatible)
===========================================

Analyzes the C code generated by FFCx and provides insights about:
- Number of assembly functions
- Quadrature rules used
- Local matrix/vector sizes
- Code complexity metrics

Usage:
    python3 analyze_ffcx_code_colab.py [filename.c]

If no filename is provided, it will search for FFCx-generated C files.
"""

import sys
import re
import os
from pathlib import Path
from collections import defaultdict

def print_section(title):
    """Print a formatted section header."""
    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)

def find_c_files():
    """Find FFCx-generated C files in current directory."""
    c_files = []
    for f in os.listdir('.'):
        if f.endswith('.c') and os.path.isfile(f):
            # Check if it looks like FFCx generated code
            with open(f, 'r') as file:
                first_lines = file.read(500)
                if 'tabulate_tensor' in first_lines or 'FFCx' in first_lines:
                    c_files.append(f)
    return c_files

def analyze_function_signatures(content):
    """Extract and analyze tabulate_tensor function signatures."""
    pattern = r'void (tabulate_tensor_\w+)\((.*?)\)'
    functions = re.findall(pattern, content, re.DOTALL)

    print_section("Assembly Functions")
    print(f"Total assembly functions found: {len(functions)}\n")

    for i, (func_name, params) in enumerate(functions[:5], 1):  # Show first 5
        print(f"{i}. {func_name}")
        # Clean up parameter formatting
        params_clean = ' '.join(params.split())
        if len(params_clean) > 60:
            params_clean = params_clean[:57] + "..."
        print(f"   Parameters: {params_clean}\n")

    if len(functions) > 5:
        print(f"   ... and {len(functions) - 5} more functions\n")

    return functions

def analyze_quadrature_rules(content):
    """Analyze quadrature rules used in the code."""
    # Find all weight arrays
    weight_pattern = r'static const double weights_\w+\[(\d+)\]\s*=\s*{([^}]+)}'
    weights = re.findall(weight_pattern, content)

    print_section("Quadrature Rules")

    if not weights:
        print("No quadrature rules found.\n")
        return

    # Analyze unique quadrature orders
    num_points = [int(w[0]) for w in weights]
    unique_points = sorted(set(num_points))

    print(f"Number of quadrature rule instances: {len(weights)}")
    print(f"Unique quadrature point counts: {unique_points}\n")

    # Show distribution
    point_counts = defaultdict(int)
    for n in num_points:
        point_counts[n] += 1

    print("Distribution of quadrature rules:")
    max_count = max(point_counts.values()) if point_counts else 1
    for n_points in sorted(point_counts.keys()):
        count = point_counts[n_points]
        bar_length = count * 20 // max(1, max_count)
        bar = "█" * bar_length
        print(f"  {n_points:2d} points: {count:3d} instances {bar}")

    # Show example weights
    if weights:
        n_points, weight_vals = weights[0]
        print(f"\nExample quadrature weights ({n_points} points):")
        vals = [float(v.strip()) for v in weight_vals.split(',')[:int(n_points)]]
        for i, val in enumerate(vals[:6]):  # Show first 6
            print(f"  w[{i}] = {val:.15f}")
        if len(vals) > 6:
            print(f"  ... and {len(vals) - 6} more weights")
    print()

def analyze_basis_functions(content):
    """Analyze basis function arrays."""
    # Pattern for basis function arrays
    pattern = r'static const double (FE\d+_C\d+_\w+)\[([^\]]+)\]\s*='
    basis_funcs = re.findall(pattern, content)

    print_section("Basis Function Arrays")

    if not basis_funcs:
        print("No basis function arrays found.\n")
        return

    print(f"Total basis function arrays: {len(basis_funcs)}\n")

    # Analyze dimensions
    for name, dims in basis_funcs[:10]:  # Show first 10
        dims_clean = dims.replace('][', '] × [')
        print(f"  {name}[{dims_clean}]")

    if len(basis_funcs) > 10:
        print(f"  ... and {len(basis_funcs) - 10} more arrays")
    print()

def analyze_loop_structures(content):
    """Analyze loop structures in assembly code."""
    # Find all for loops
    loop_pattern = r'for\s*\(int\s+(\w+)\s*=\s*0;\s*\1\s*<\s*(\d+);'
    loops = re.findall(loop_pattern, content)

    print_section("Loop Structures")

    if not loops:
        print("No loops found.\n")
        return

    # Analyze loop variables and bounds
    loop_vars = defaultdict(list)
    for var, bound in loops:
        loop_vars[var].append(int(bound))

    print(f"Total loops found: {len(loops)}\n")

    print("Common loop patterns:")
    for var in sorted(loop_vars.keys()):
        bounds = loop_vars[var]
        unique_bounds = sorted(set(bounds))
        print(f"  for (int {var} = 0; {var} < n; ...)")
        print(f"    Used {len(bounds)} times")
        print(f"    Bounds: {unique_bounds}")
        if var == 'iq':
            print(f"    → Quadrature point loops")
        elif var in ['i', 'j']:
            print(f"    → DOF loops (matrix assembly)")
        elif var == 'ic':
            print(f"    → Coordinate/component loops")
        print()

def analyze_code_size(filepath):
    """Analyze code size metrics."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    print_section("Code Size Metrics")

    total_lines = len(lines)
    code_lines = sum(1 for line in lines if line.strip() and not line.strip().startswith('//'))
    comment_lines = sum(1 for line in lines if line.strip().startswith('//'))
    blank_lines = total_lines - code_lines - comment_lines

    print(f"Total lines:        {total_lines:6d}")
    print(f"Code lines:         {code_lines:6d} ({100*code_lines/total_lines:.1f}%)")
    print(f"Comment lines:      {comment_lines:6d} ({100*comment_lines/total_lines:.1f}%)")
    print(f"Blank lines:        {blank_lines:6d} ({100*blank_lines/total_lines:.1f}%)")

    # Estimate complexity (avoid backslashes in f-strings)
    content = ''.join(lines)
    for_for_pattern = r'for.*for'
    if_pattern = r'\bif\s*\('
    mult_pattern = r'\*'
    add_pattern = r'\+'

    nested_loops = len(re.findall(for_for_pattern, content, re.DOTALL))
    conditionals = len(re.findall(if_pattern, content))
    multiplications = len(re.findall(mult_pattern, content))
    additions = len(re.findall(add_pattern, content))

    print(f"\nComplexity indicators:")
    print(f"  Nested loops:     {nested_loops:6d}")
    print(f"  Conditionals:     {conditionals:6d}")
    print(f"  Multiplications:  {multiplications:6d}")
    print(f"  Additions:        {additions:6d}")
    print()

def analyze_memory_usage(content):
    """Estimate memory usage from static arrays."""
    # Find all static arrays with sizes
    array_pattern = r'static const double \w+\[([\d\]\[]+)\]'
    arrays = re.findall(array_pattern, content)

    print_section("Memory Usage Estimate")

    total_elements = 0
    for dims_str in arrays:
        # Parse dimensions
        dims = [int(d) for d in dims_str.replace('][', ' ').replace('[', '').replace(']', '').split()]
        elements = 1
        for d in dims:
            elements *= d
        total_elements += elements

    # Each double is 8 bytes
    total_bytes = total_elements * 8

    print(f"Static array elements: {total_elements:,}")
    print(f"Estimated memory:      {total_bytes:,} bytes")
    print(f"                       {total_bytes/1024:.1f} KB")
    if total_bytes > 1024*1024:
        print(f"                       {total_bytes/(1024*1024):.2f} MB")
    print()

def generate_summary(content, filepath):
    """Generate an overall summary."""
    print_section("Summary")

    # Count key elements
    num_functions = len(re.findall(r'void tabulate_tensor_', content))
    num_static_arrays = len(re.findall(r'static const double', content))
    num_loops = content.count('for (')

    file_size = Path(filepath).stat().st_size

    print(f"File: {filepath}")
    print(f"File size: {file_size:,} bytes ({file_size/1024:.1f} KB)")
    print(f"\nCode structure:")
    print(f"  Assembly functions:   {num_functions}")
    print(f"  Static data arrays:   {num_static_arrays}")
    print(f"  Loop constructs:      {num_loops}")
    print(f"\nThis code was automatically generated by FFCx from UFL forms.")
    print(f"It implements optimized finite element assembly kernels.")
    print()

def main():
    """Main analysis function."""

    # Determine which file to analyze
    filepath = None

    if len(sys.argv) >= 2:
        filepath = sys.argv[1]
    else:
        # Search for FFCx-generated C files
        print("No file specified. Searching for FFCx-generated C files...")
        c_files = find_c_files()

        if not c_files:
            print("\nError: No FFCx-generated C files found in current directory.")
            print("\nUsage: python3 analyze_ffcx_code_colab.py <ffcx_generated.c>")
            print("\nExample:")
            print("  python3 analyze_ffcx_code_colab.py mass_matrix.c")
            return 1

        print(f"\nFound {len(c_files)} FFCx-generated C file(s):")
        for i, f in enumerate(c_files, 1):
            size = os.path.getsize(f)
            print(f"  {i}. {f} ({size:,} bytes)")

        # Use the first file found
        filepath = c_files[0]
        print(f"\nAnalyzing: {filepath}")

    if not Path(filepath).exists():
        print(f"Error: File not found: {filepath}")
        return 1

    # Read the generated C code
    try:
        with open(filepath, 'r') as f:
            content = f.read()
    except Exception as e:
        print(f"Error reading file: {e}")
        return 1

    print("=" * 70)
    print(" FFCx Generated Code Analysis")
    print("=" * 70)

    # Run all analyses
    try:
        analyze_function_signatures(content)
        analyze_quadrature_rules(content)
        analyze_basis_functions(content)
        analyze_loop_structures(content)
        analyze_code_size(filepath)
        analyze_memory_usage(content)
        generate_summary(content, filepath)
    except Exception as e:
        print(f"\nError during analysis: {e}")
        import traceback
        traceback.print_exc()
        return 1

    print("=" * 70)
    print(" Analysis Complete")
    print("=" * 70)

    return 0

if __name__ == "__main__":
    sys.exit(main())

No file specified. Searching for FFCx-generated C files...

Found 1 FFCx-generated C file(s):
  1. ffcx_examples.c (22,885 bytes)

Analyzing: ffcx_examples.c
 FFCx Generated Code Analysis

 Assembly Functions
Total assembly functions found: 4

1. tabulate_tensor_integral_29a541bd7d5bdf80771174dde995f0d3631e75be_triangle
   Parameters: double* restrict A, const double* restrict w, const doubl...

2. tabulate_tensor_integral_57d8310a3ad4f190bcd6180f39ea8448e88e56bb_triangle
   Parameters: double* restrict A, const double* restrict w, const doubl...

3. tabulate_tensor_integral_a7e95df848e1f0d8306fa59c68c7ccb507fd6eee_triangle
   Parameters: double* restrict A, const double* restrict w, const doubl...

4. tabulate_tensor_integral_dd4f2a76f1db1a65f63a3f28ed1f91cd444aa672_triangle
   Parameters: double* restrict A, const double* restrict w, const doubl...


 Quadrature Rules
Number of quadrature rule instances: 4
Unique quadrature point counts: [1, 3, 6]

Distribution of quadrature rules:
 